In [3]:
import ee
import os
import json
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np


In [5]:
PROJECT_ROOT = Path().resolve().parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [3]:
ee.Authenticate()



Successfully saved authorization token.


In [38]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [151]:
import importlib
import data_utils
import gee_utils

importlib.reload(data_utils)
importlib.reload(gee_utils)

<module 'gee_utils' from '/Users/bensutton/Projects/dissertation-code/src/gee_utils.py'>

In [36]:

sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import create_images_for_all_locations, get_samples, export_patches, export_awei_p95_all_locations
from data_utils import make_padded_bbox_all_location, clean_labelled_data


In [11]:
# Want to upload the sites.json to ee and then create image collection for each, clip to the bbox, need to select the dates for each location
# Could save the location in the json or just use the date from the points? 

EE_PROJECT = os.environ["EE_PROJECT"]
site_fp = PROJECT_ROOT / "configs" / "sites.json"
label_fp = PROJECT_ROOT / "configs" / "labels.gpkg"
cleaned_label_fp = PROJECT_ROOT / "configs" / "cleaned_labels.gpkg"

In [36]:
clean_labelled_data(label_fp= label_fp, cleaned_label_fp = cleaned_label_fp)

Saved a cleaned version of /Users/bensutton/Projects/dissertation-code/configs/labels.gpkg as /Users/bensutton/Projects/dissertation-code/configs/cleaned_labels.gpkg


In [12]:
# test the created cleaned labels geopackage


labels_gdf = gpd.read_file(cleaned_label_fp)

labels_gdf.head()



,label_id,longitude,latitude,location,obs_date,comparison_dates_used,s2_target_image_id,s2_old_image_id,s1_image_id,class_label,notes,created_at,updated_at,created_by,class_int,lc,geometry
0,HA0001_20210121,27.823149,-25.752589,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:04.027316+00:00,2026-05-14T14:19:04.027316+00:00,bensutton,3,1,POINT (27.82315 -25.75259)
1,HA0002_20210121,27.807339,-25.760474,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:06.695977+00:00,2026-05-14T14:19:06.695977+00:00,bensutton,3,1,POINT (27.80734 -25.76047)
2,HA0003_20210121,27.809334,-25.755411,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:10.017332+00:00,2026-05-14T14:19:10.017332+00:00,bensutton,3,1,POINT (27.80933 -25.75541)
3,HA0004_20210121,27.805043,-25.758155,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:13.638342+00:00,2026-05-14T14:19:13.638342+00:00,bensutton,3,1,POINT (27.80504 -25.75816)
4,HA0005_20210121,27.819760,-25.762774,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:18.937682+00:00,2026-05-14T14:19:18.937682+00:00,bensutton,3,1,POINT (27.81976 -25.76277)


In [14]:
print(len(labels_gdf))
labels_gdf.groupby(["location", "class_label"])["lc"].count().reset_index(name="count")

16192


,location,class_label,count
0,Hartbeespoort,LEV,509
1,Hartbeespoort,floating_plants,1002
2,Hartbeespoort,open_water,440
3,Hartbeespoort,surface_algae,81
4,Inle,LEV,500
5,Inle,floating_plants,1000
6,Inle,open_water,500
7,Mula,LEV,576
8,Mula,floating_plants,1003
9,Mula,open_water,493


In [16]:

labels_gdf.groupby(["location"])["lc"].count().reset_index(name="count")

,location,count
0,Hartbeespoort,2032
1,Inle,2000
2,Mula,2072
3,RawaPening,2004
4,Rodman,2042
5,Valsequillo,2040
6,Vembanad,2002
7,Winam,2000


In [40]:
# Make a bounding box for each site based on the total bounds of the labelled points and add a padding

make_padded_bbox_all_location(sites_file= site_fp, project_root= PROJECT_ROOT)

Created padded bbox from the points files for Vembanad, Winam, Inle, Hartbeespoort, Mula, RawaPening, Rodman, Valsequillo


In [ ]:
# Memomry limits made it neccesary to create an AWEI_p95 (with the 95th percentile for AWEIsh, 2019-2025) image that is saved to assets for each location.
# After running the code to export use the returned task list areto ensure exports are completed before running create_images_for_all_locations()

task_list = export_awei_p95_all_locations(ee_project= EE_PROJECT, sites_file= site_fp)

In [ ]:
for task in task_list:
    print(task.status())

In [ ]:
image_collection = create_images_for_all_locations(sites_file= site_fp, cleaned_label_fp=cleaned_label_fp, ee_project=EE_PROJECT, clip = False)

In [147]:
print(image_collection.first().bandNames().getInfo())

['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'AWEIp95']


In [ ]:
# Due to memory limits it is not possible to convert all the sampled points to a data frame, therefore the sampled points for each location 
# is export to drive as a CSV separately


# Trying to save all without aweip95

get_samples(merged_ic=image_collection,cleaned_label_fp= cleaned_label_fp)

In [ ]:
for task in ee.batch.Task.list():
    status = task.status()
    desc = status.get("description", "")

    if desc.startswith("sampled_points_"):
        print(desc, status["state"])
        print("EECU seconds:", status.get("batch_eecu_usage_seconds"))
        if status["state"] == "FAILED":
            print(status.get("error_message"))

In [18]:
# Load location samples from csvs in outputs/sampled_points_by_location, combine into a single dataframe and save 

samples_outpath = PROJECT_ROOT / "outputs" / "sample_points.csv"

with open(site_fp) as f:
    sites= json.load(f)

location_list = list(sites.get("sites", {}).keys())

list_of_location_dfs = []

for location in location_list:
    location_samples_fp = PROJECT_ROOT / "outputs/sampled_points_by_location/sampled_points" / f"sampled_points_{location}.csv" 

    location_df = pd.read_csv(location_samples_fp)

    list_of_location_dfs.append(location_df)

all_samples = pd.concat(list_of_location_dfs, ignore_index= True)



all_samples.to_csv(samples_outpath, index= False)



-------------------------
Some points are missing between labels.phkg ad the sampled points. Approx 170, trying to find out why

In [ ]:
# check len of inle 

inle_fp = PROJECT_ROOT / "outputs/sampled_points_by_location/sampled_points" / f"sampled_points_inle.csv"

inle_points = pd.read_csv(inle_fp)
len(inle_points)


1830

In [33]:
inle_no_awei_fp = PROJECT_ROOT / "outputs/sampled_points_by_location/sampled_points_no_AWEI" / f"sampled_points_inle.csv"

inle_no_awei = pd.read_csv(inle_no_awei_fp)

print(len(inle_no_awei))
inle_no_awei.head()

1830


,system:index,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,class_int,class_label,label_id,latitude,lc,location,longitude,obs_date,.geo
0,0_0_0,706,303,267,436,209,629,1857,2287,2536,2440,3,floating_plants,IN0001_20191020,20.578856,1,Inle,96.896985,2019-10-20,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,0_1_0,466,213,236,402,244,604,1466,1606,1776,1733,3,floating_plants,IN0002_20191020,20.580544,1,Inle,96.896685,2019-10-20,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,0_2_0,414,179,261,421,227,574,1213,1324,2162,1515,3,floating_plants,IN0003_20191020,20.578515,1,Inle,96.896213,2019-10-20,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,0_3_0,597,246,279,513,254,599,1686,2050,2810,2264,3,floating_plants,IN0004_20191020,20.578093,1,Inle,96.898658,2019-10-20,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,0_4_0,736,304,251,418,223,701,2021,2397,2412,2521,3,floating_plants,IN0005_20191020,20.577711,1,Inle,96.897285,2019-10-20,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [19]:
# Number of samples per class per location
all_samples_fp = PROJECT_ROOT / "outputs" / "sample_points.csv"
all_samples = pd.read_csv(all_samples_fp)

print(len(all_samples))
all_samples.groupby(["location"])["lc"].count().reset_index(name="count")

16019


,location,count
0,Hartbeespoort,2032
1,Inle,1830
2,Mula,2072
3,RawaPening,2002
4,Rodman,2042
5,Valsequillo,2040
6,Vembanad,2002
7,Winam,1999


In [23]:
# Per (location, obs_date) point counts: labels vs sampled
lab_counts = (
    labels_gdf.groupby(["location", "obs_date"])
    .size()
    .rename("n_labels")
)
samp_counts = (
    all_samples.groupby(["location", "obs_date"])
    .size()
    .rename("n_sampled")
)

comparison = (
    pd.concat([lab_counts, samp_counts], axis=1)
    .fillna(0)
    .astype(int)
)
comparison["lost"] = comparison["n_labels"] - comparison["n_sampled"]

# Only the date/location combos where points went missing
lost = comparison[comparison["lost"] != 0].sort_values("lost", ascending=False)

print(f"Total labels:  {comparison['n_labels'].sum()}")
print(f"Total sampled: {comparison['n_sampled'].sum()}")
print(f"Total lost:    {comparison['lost'].sum()}\n")

# Per-location summary
print(comparison.groupby(level="location")["lost"].sum().loc[lambda s: s != 0])
print()

# Detailed: which dates lose points
lost

Total labels:  16192
Total sampled: 16019
Total lost:    173

location
Inle          170
RawaPening      2
Winam           1
Name: lost, dtype: int64



n_labels  n_sampled  lost
location   obs_date                             
Inle       2023-01-17       100         52    48
           2020-10-14       300        257    43
           2021-10-09       260        224    36
           2020-12-13       180        155    25
           2021-12-08        80         63    17
           2025-05-06        40         39     1
RawaPening 2019-07-28       240        239     1
           2024-08-10       202        201     1
Winam      2024-11-26       300        299     1

In [25]:
import json
from pathlib import Path
from shapely.geometry import box

# --- identify the lost points by label_id set difference ---
lost_ids = set(labels_gdf["label_id"]) - set(all_samples["label_id"])
lost_pts = labels_gdf[labels_gdf["label_id"].isin(lost_ids)].to_crs("EPSG:4326").copy()
print(f"{len(lost_pts)} lost points\n")

# --- load the stored bboxes from sites.json ---
with open(PROJECT_ROOT / "configs" / "sites.json", encoding="utf-8") as f:
    sites = json.load(f)

def bbox_poly(loc):
    coords = sites["sites"][loc]["bbox"]["coordinates"][0]
    xs = [c[0] for c in coords]; ys = [c[1] for c in coords]
    return box(min(xs), min(ys), max(xs), max(ys))

# --- test each lost point against its location's bbox ---
def inside_bbox(row):
    return bbox_poly(row["location"]).contains(row.geometry)

lost_pts["inside_bbox"] = lost_pts.apply(inside_bbox, axis=1)

print("Lost points OUTSIDE their bbox (AWEIp95 region) — would be dropped:")
print(lost_pts.groupby("location")["inside_bbox"].apply(lambda s: (~s).sum()))
print("\nLost points INSIDE bbox (masked AWEIp95 or cloud, not a footprint issue):")
print(lost_pts.groupby("location")["inside_bbox"].apply(lambda s: s.sum()))

173 lost points

Lost points OUTSIDE their bbox (AWEIp95 region) — would be dropped:
location
Inle          0
RawaPening    0
Winam         0
Name: inside_bbox, dtype: int64

Lost points INSIDE bbox (masked AWEIp95 or cloud, not a footprint issue):
location
Inle          170
RawaPening      2
Winam           1
Name: inside_bbox, dtype: int64


In [27]:
# Lost points: in labels but not in samples
lost_ids = set(labels_gdf["label_id"]) - set(all_samples["label_id"])

lost_pts = (
    labels_gdf.loc[
        labels_gdf["label_id"].isin(lost_ids),
        ["label_id", "location", "class_label"],
    ]
    .sort_values(["location", "class_label", "label_id"])
)

out_fp = PROJECT_ROOT / "outputs" / "lost_points.csv"
out_fp.parent.mkdir(parents=True, exist_ok=True)
lost_pts.to_csv(out_fp, index=False)

print(f"Saved {len(lost_pts)} lost points to {out_fp}")
lost_pts

Saved 173 lost points to /Users/bensutton/Projects/dissertation-code/outputs/lost_points.csv


,label_id,location,class_label
11494,IN0074_20211208,Inle,LEV
11495,IN0075_20211208,Inle,LEV
10486,IN0077_20230117,Inle,LEV
10487,IN0078_20230117,Inle,LEV
11499,IN0079_20211208,Inle,LEV
...,...,...,...
11182,IN0236_20201014,Inle,open_water
11205,IN0259_20201014,Inle,open_water
14524,RA0158_20190728,RawaPening,LEV
2257,RA0195_20240810,RawaPening,LEV


In [39]:
merged_ic = create_images_for_all_locations(sites_file=site_fp, cleaned_label_fp= cleaned_label_fp, ee_project= EE_PROJECT)

Created image collection for: Vembanad
Created image collection for: Winam
Created image collection for: Inle
Created image collection for: Hartbeespoort
Created image collection for: Mula
Created image collection for: RawaPening
Created image collection for: Rodman
Created image collection for: Valsequillo


In [ ]:
import ee, pandas as pd

# points present in labels but missing from the samples
lost_ids = set(labels_gdf["label_id"]) - set(all_samples["label_id"])
print(f"{len(lost_ids)} lost label_ids")

lost_geo = labels_gdf[labels_gdf["label_id"].isin(lost_ids)].to_crs("EPSG:4326").copy()
lost_geo["obs_date"] = lost_geo["obs_date"].astype(str)

records = []
for (loc, date), grp in lost_geo.groupby(["location", "obs_date"]):
    image = (merged_ic
             .filter(ee.Filter.eq("location", loc))
             .filter(ee.Filter.eq("obs_date", date))
             .first())

    feats = [ee.Feature(ee.Geometry.Point([r.geometry.x, r.geometry.y]),
                        {"label_id": r.label_id}) for r in grp.itertuples()]
    fc = ee.FeatureCollection(feats)
    proj = image.select("B2").projection()

    # (A) does sampleRegions actually return these points?
    got = {f["properties"]["label_id"]
           for f in image.sampleRegions(collection=fc, properties=["label_id"],
                                        scale=10, projection=proj,
                                        geometries=False).getInfo()["features"]}

    # (B) per-band mask at each point (mask image is never itself masked → nothing dropped)
    mask_vals = image.mask().reduceRegions(
        collection=fc, reducer=ee.Reducer.first(),
        scale=10).getInfo()["features"]

    for f in mask_vals:
        p = f["properties"]
        records.append({
            "label_id": p["label_id"], "location": loc, "obs_date": date,
            "returned_by_sampleRegions": p["label_id"] in got,
            "B2_valid": p.get("B2"), "B8_valid": p.get("B8"),
            "B11_valid": p.get("B11"),
        })

diag = pd.DataFrame(records)
print(diag["returned_by_sampleRegions"].value_counts())
print(diag[["B2_valid", "B8_valid", "B11_valid"]].describe())
diag

173 lost label_ids


-------------------------------------------------------------------------------------
Sample the patches from the labels


In [ ]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 15)

In [ ]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 31)

In [134]:
asset_id = 'projects/' + EE_PROJECT + '/assets/awei_p95_Winam'

AWEI_image = ee.Image(asset_id)

display('bands', AWEI_image.bandNames())

'bands'

In [ ]:
stats = AWEI_image.reduceRegion(
    reducer=ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
    geometry=AWEI_image.geometry(),
    scale=10,      # use the appropriate resolution
    maxPixels=1e13
)

print(stats.getInfo())

{'AWEIp95_max': 12159.125, 'AWEIp95_mean': 955.1331094299671, 'AWEIp95_min': -5204}


In [ ]:
# For combining GEOJSON

feature_list = []


location_list = ["Hartbeespoort", "Rodman","Mula", "Inle", "Vembanad", "Valsequillo", "RawaPening"]

for location in location_list:
    location_fp = PROJECT_ROOT / "outputs/31px_patches" / f"sampled_31_pixel_patches_for_{location}.geojson"
    with open(location_fp, "r") as f:
        patches = json.load(f)

        for feature in patches["features"]:
            feature["id"] = feature["properties"]["label_id"]
            feature_list.append(feature)

combined_dict = {"type": "FeatureCollection", "features": feature_list}

outpath = PROJECT_ROOT / "outputs/15px_patches" / "combined_15px_patches.geojson"

with open(outpath, 'w') as f:
    json.dump(combined_dict, f)